In [ ]:
import numpy as np
import psfsim
import matplotlib.pyplot as plt
import matplotlib as mpl


import importlib

importlib.reload(psfsim)
from psfsim.psfobject import PSFObject

In [ ]:
def downsample_2d_image(image, pixel_size=8):  # ONLY WORKS ON 2D IMAGES
    subpixel_side = np.arange(pixel_size)
    subpixel_cols, subpixel_rows = np.meshgrid(subpixel_side, subpixel_side)

    dsamp_size = image.shape[0] // pixel_size
    pixels_side = np.arange(dsamp_size) * pixel_size
    pixels_cols, pixels_rows = np.meshgrid(pixels_side, pixels_side)

    pixels_rows = pixels_rows[:, :, np.newaxis]  # promote to 3D
    pixels_cols = pixels_cols[:, :, np.newaxis]

    idx_arr_subpixel_rows = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_rows.flatten())
    idx_arr_subpixel_cols = np.full((dsamp_size, dsamp_size, pixel_size**2), subpixel_cols.flatten())

    idx_arr_rows = idx_arr_subpixel_rows + pixels_rows
    idx_arr_cols = idx_arr_subpixel_cols + pixels_cols

    pixelated_image = image[idx_arr_rows, idx_arr_cols]
    downsampled_image = np.mean(pixelated_image, axis=-1)
    return downsampled_image

In [ ]:
newobj = PSFObject(
    9,
    0,
    0,
    wavelength=1.25,
    postage_stamp_size=32,
    use_postage_stamp_size=None,
    extra_aberrations=(None, None, 0.5, 0.8, 0.3),
)

# newobj = PSFObject(
#     9,0,0, wavelength=1.25, postage_stamp_size=32, use_postage_stamp_size=None,
#     extra_aberrations = ( None, None, 0.5, None, None ), ovsamp=8
# )

# newobj = PSFObject(
#     9,0,0, wavelength=1.25, postage_stamp_size=32, use_postage_stamp_size=None,
#     extra_aberrations = ( None, None, None, None, 1. )
# )

newobj.get_optical_psf()

plt.imshow(downsample_2d_image(np.log10(np.abs(newobj.Optical_PSF))))
plt.colorbar()

Start from the true center (data coords 1024, 1024) for now and draw lines at each angle. Sum up the PSF along the lines and find the maxima above some cutoff. Once we have the angles we can compare to the ideal image. Can also jiggle the center around a bit in the future to find the best possible fitting line. We will use a 2d rotation matrix to get the lines at each angle for now. Bresenham's algorithm could also be used.


In [ ]:
# newobj.Optical_PSF.shape


# angle in degrees
# rotation matrix
# to cut down on difficulty in Bresenham's (if needed) we only compute lines in the first octant (0 < angle < 45deg)
# and then rotate all the points via a rotation matrix

# imshow displays images like you're reading off a table: top left is 0,0 and elements are indexed row,column. The method I thought of works but
# I had it flipped. We are drawing lines on this table. The angles are as we see them.


def line_coords(angle, bound, center=np.array((0, 0))):
    # since (0, 0) in data coords is top left instead of bottom left we must use clockwise rotations to get
    # a visually ccl rotation. However we will take a transpose later anyways so the rotation matrix def is ccl, as expected. How quaint
    angle = np.deg2rad(angle)
    # print( angle )
    line = np.zeros((2, bound))
    line[0] = np.arange(bound)

    rotation = np.array(
        (
            (np.cos(angle), -np.sin(angle)),
            (np.sin(angle), np.cos(angle)),
        )
    )

    if len(rotation.shape) > 2:
        line = line.reshape((1, 2, bound))

    # print( line.shape )
    # print( rotation.T.shape )

    return np.int32(rotation.T @ line) + np.int32(np.reshape(center, (2, 1)))


angle = 112  # degrees
bound = 2048 // 2


def find_spikes(image, step, bound=1024, center=np.array((0, 0)), exclude_center=False, verbose=False):
    # TODO: Prominence and Spike width
    # prominence filtering is the next step up in analyzing the sums plot to find spikes. While it is true that small aberrations
    # are easy to analyze, spike width and prominence (brightness vs background) may prove to be valuable pieces of information
    # later on. Also I would like to figure out how to find the spikes in difficult situations.
    #
    #
    #

    angles = np.linspace(0, 360, num=np.int32(360.0 / step), endpoint=False)

    lines = line_coords(angles, bound, center)
    image_analyze = (
        image.T
    )  # imshow will display the image as row,col with 0,0 in top left. For our analysis we want (col,row).

    line_vals = image_analyze[lines[:, 0], lines[:, 1]]
    print(line_vals.shape)
    if exclude_center:
        sums = np.sum(line_vals[:, (bound // 2) :], axis=-1)
    else:
        sums = np.sum(line_vals, axis=-1)
    """
    This simple midpoint formula is the first attempt at discriminating maxima from each other. If we want to find less well-defined spikes
    this could become more challenging.
    """
    cutoff = (np.min(sums) + np.max(sums)) / 2
    spike_indices = np.where(sums > cutoff)[0]
    spike_angles = angles[spike_indices]
    # print( np.diff( spike_angles ) )

    # try 3 degrees to tell differing spikes apart for now
    spike_angle_discriminator = 3.0
    borders = np.where(np.diff(spike_angles) > spike_angle_discriminator)[0]
    spike_groups = np.split(spike_angles, borders + 1)
    # print( spike_groups )

    # unfortunately this for loop is necessary unless theres a convenient package for jagged arrays
    spike_list = np.zeros(len(spike_groups))
    for i in np.arange(len(spike_groups)):
        spike_list[i] = np.median(spike_groups[i])

    # print( spike_list )
    if verbose:
        return sums, spike_angles, cutoff, spike_list
    else:
        return spike_list


def draw_ray(ax, angle, bound, center=np.array((0, 0)), **kwargs):
    line = line_coords(angle, bound, center)
    ax.plot(line[0], line[1], **kwargs)
    return


def boxcar_average(arr, window_size=5):
    # will default to doing what cumsum does, which is to flatten the array. changing this is really easy
    assert window_size <= arr.shape[0]
    sums = np.cumsum(arr)
    return (sums[window_size - 1 :] - np.concatenate(((0,), sums))[:-window_size]) / window_size

In [ ]:
boxcar_average(np.arange(1, 11), window_size=2)

In [ ]:
line = line_coords(angle, 900, center=(1024, 1050))
# print( line )

plt.ion()

fig, ax = plt.subplots(figsize=(12, 5.5), ncols=2)

psf_image = np.log10(np.abs(newobj.Optical_PSF))
im = ax[0].imshow(psf_image)
fig.colorbar(im, ax=ax[0])

im = ax[1].imshow(psf_image)
fig.colorbar(im, ax=ax[1])


sums_auto, spike_angles_auto, cutoff_auto, spike_list = find_spikes(
    psf_image, 0.1, center=(1024, 1024), verbose=True
)


# The solution is to take the transpose of the thing we display
psf_image = psf_image.T
# print( psf_image.shape )
# ax.plot( line[0], line[1], color="C3" );
psf_vals = psf_image[line[0], line[1]]
# print( np.sum( psf_vals ) )
angles = np.arange(360)
lines = line_coords(angles, 1024, center=(1024, 1024))
# print( lines.shape )

many_psf_vals = psf_image[lines[:, 0], lines[:, 1]]


sums = np.sum(many_psf_vals, axis=-1)
print(np.median(sums))
fig2, ax2 = plt.subplots(figsize=(6, 16), nrows=3)
ax2[0].plot(sums_auto)

# ax2[0].set_xlim( 1000, 1350 )

ax2[1].plot(many_psf_vals[23])
ax2[2].plot(many_psf_vals[10])
cutoff = (np.max(sums) + np.min(sums)) / 2
# ax2[0].axhline( cutoff, color='C3', ls='--' )
# ax2[0].axhline( np.median( sums ), ls='--' )
# print( cutoff )
indices = np.where(sums > cutoff)[0]

window_size = 5
boxcar_sums = boxcar_average(sums, window_size=window_size)
# ax2[0].plot( np.concatenate( ( np.full( window_size - 1, sums[0] ), boxcar_sums ) ) )

# ax.plot( lines[ 10, 0 ], lines[ 10, 1 ], color="C4" )

for spike in spike_list:
    draw_ray(ax[0], spike, bound, center=(1024, 1024), color="C3")
    draw_ray(ax[1], spike, bound, center=(1024, 1024), color="C3")

print(spike_list)

# angles = ''
# for idx in indices:
#     angles += f'{idx} '
#     ax.plot( lines[idx, 0], lines[idx, 1], color="C3" )

# print( angles )
unfocused_spikes = spike_list

# ax[0].set_xlim( 1004, 1044 )
# ax[0].set_ylim( 1044, 1004 )

ax[1].set_xlim(1064, 1124)
ax[1].set_ylim(1074, 1014)

In [ ]:
l = 0
r = 200

fig, ax = plt.subplots(figsize=(6, 12), nrows=2)
ax[0].plot(sums_auto)
ax[0].set_xlim(l, r)

ax[1].plot(np.diff(sums_auto))
ax[1].set_xlim(l, r)

In [ ]:
find_spikes?

In [ ]:
# print( lines[10,0] )
# print( lines[10,1] )

fig, ax = plt.subplots()

psf_image = np.log10(np.abs(newobj.Optical_PSF))
im = ax.imshow(psf_image, cmap="plasma")
fig.colorbar(im, ax=ax)

range_low = 400
range_high = 650

ax.plot(lines[10, 0, range_low:range_high], lines[10, 1, range_low:range_high], color="C7")

for angle in spike_angles_auto:
    draw_ray(ax, angle, bound, center=(1024, 1024), color="C9")

# for idx in indices:
#     # print( idx )
#     ax.plot( lines[idx, 0, range_low:range_high ], lines[idx, 1, range_low:range_high ], color="C9" )


# ax.set_xlim( 1300, 1350 )
# ax.set_ylim( 950, 900 )

In [ ]:
print(lines[23, 0, 500], lines[23, 1, 500])
print(psf_image[1484, 855])
print(psf_image[857, 1492])

In [ ]:
fig2, ax2 = plt.subplots()

psf_image = np.log10(np.abs(newobj.Optical_PSF))
im = ax2.imshow(psf_image, cmap="plasma")
fig2.colorbar(im, ax=ax2)

range_low = 400
range_high = 650

ax2.plot(lines[10, 0, range_low:range_high], lines[10, 1, range_low:range_high], color="C0")

for idx in indices:
    # print( idx )
    ax2.plot(lines[idx, 0, range_low:range_high], lines[idx, 1, range_low:range_high], color="C0")


# ax2.set_xlim( 1375, 1650 )
# ax2.set_ylim( 1000, 750 )

# ax2.set_xlim( 1520, 1530 )
# ax2.set_ylim( 965, 955 )

ax2.set_xlim(1488, 1492)
ax2.set_ylim(860, 856)

In [ ]:
print(psf_image.shape)

In [ ]:
newobj2 = PSFObject(9, 0, 0, wavelength=1.25, postage_stamp_size=32, use_postage_stamp_size=None)
newobj2.get_optical_psf()

In [ ]:
newobj3 = PSFObject(
    9,
    0,
    0,
    wavelength=1.25,
    postage_stamp_size=32,
    use_postage_stamp_size=None,
    extra_aberrations=(None, None, 0.1),
)
newobj3.get_optical_psf()

In [ ]:
newobj4 = PSFObject(
    9,
    0,
    0,
    wavelength=1.25,
    postage_stamp_size=32,
    use_postage_stamp_size=None,
    extra_aberrations=(None, None, -0.1),
)
newobj4.get_optical_psf()

In [ ]:
newobj6 = PSFObject(
    9,
    0,
    0,
    wavelength=1.25,
    postage_stamp_size=32,
    use_postage_stamp_size=None,
    extra_aberrations=(None, None, 0.3),
)
newobj6.get_optical_psf()

In [ ]:
fig, ax = plt.subplots()

ideal_img = np.log10(np.abs(newobj2.Optical_PSF))
im = ax.imshow(ideal_img)
fig.colorbar(im, ax=ax)

ideal_spikes = find_spikes(ideal_img, 0.1, center=(1024, 1024))

for spike in ideal_spikes:
    draw_ray(ax, spike, bound, center=(1024, 1024), color="C3")

print(ideal_spikes)

In [ ]:
fig, ax = plt.subplots()

img3 = np.log10(np.abs(newobj3.Optical_PSF))
im = ax.imshow(img3)
fig.colorbar(im, ax=ax)

spikes_3 = find_spikes(img3, 0.1, center=(1024, 1024))

for spike in spikes_3:
    draw_ray(ax, spike, bound, center=(1024, 1024), color="C3")

print(spikes_3)

In [ ]:
fig, ax = plt.subplots()

img4 = np.log10(np.abs(newobj4.Optical_PSF))
im = ax.imshow(img4)
fig.colorbar(im, ax=ax)

spikes_4 = find_spikes(img4, 0.1, center=(1024, 1024))

for spike in spikes_4:
    draw_ray(ax, spike, bound, center=(1024, 1024), color="C3")

In [ ]:
img6 = np.log10(np.abs(newobj6.Optical_PSF))
spikes_6 = find_spikes(img6, 0.1, center=(1024, 1024))

Surprisingly this seems to work fairly well even for small focus parameters. For 0.01 focus we can see that there is a difference but it is quite small. Half of the spikes are less than 15 arcminutes off!

In [ ]:
n = np.arange(np.prod(ideal_spikes.shape))

# plt.scatter( n, unfocused_spikes )
# plt.scatter( n, ideal_spikes )
plt.scatter(n, unfocused_spikes - ideal_spikes, label="0.5")
# plt.scatter( n, spikes_3 - ideal_spikes, label='0.1' )
# plt.scatter( n, spikes_4 - ideal_spikes, label='-0.1' )
# plt.scatter( n, spikes_6 - ideal_spikes, label='0.3' )
plt.title("Difference in diffraction spike angle vs. focused PSF")
plt.ylabel("Angle (degrees)")
plt.legend()
plt.axhline(0, color="C3", ls="--")

In [ ]:
# chisq values, could maybe plot these vs focus parameter. Doing so would require generating a bunch of psfs.


# steps of 0.02 focus parameter up to 0.5

begin = 0.02
step = 0.02
end = 0.1
N = np.int32(end / step)
foci = np.linspace(begin, end, N)
chisq = []

for focus in foci:
    newobj5 = PSFObject(
        9, 0, 0, wavelength=1.25, postage_stamp_size=32, use_postage_stamp_size=None, add_focus=focus
    )
    newobj5.get_optical_psf()

    img = np.log10(np.abs(newobj5.Optical_PSF))
    spikes_5 = find_spikes(img, 0.5, center=(1024, 1024))

    resid = spikes_5 - ideal_spikes
    chisq += [np.sum(resid**2)]

print(chisq)
# resid_1 = unfocused_spikes - ideal_spikes
# resid_2 = spikes_3 - ideal_spikes
# resid_3 = spikes_4 - ideal_spikes

# print( np.sum( resid_1**2 ) )
# print( np.sum( resid_2**2 ) )
# print( np.sum( resid_3**2 ) )

In [ ]:
a = np.arange(len(chisq))
plt.scatter(foci, np.array(chisq))